<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/bonus/notebooks/5_annotation_and_validation_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Bonus: Annotation and Validation · Solutions

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every ✏️ exercise is filled in with one possible answer and a
short comment. Your numbers will differ a little from run to run, so read the comments as "what
to look for", not as exact results.

Every automated measure in this course (a dictionary, a classifier, a fine-tuned BERT, an LLM) is
only as credible as its agreement with careful human judgment. *Validate, validate, validate*
(Grimmer & Stewart, 2013). This notebook is about producing that human judgment well, and using it
correctly:

- writing a codebook and drawing a sample to annotate
- measuring agreement between coders: percent agreement, Cohen's kappa, Krippendorff's alpha, and
  why percent agreement misleads
- learning from disagreements
- validating a classifier against the human "gold standard", with honest uncertainty
- how many documents you need to annotate
- why a decent classifier can still give biased *proportions*, and a simple correction

We simulate the coders, so we know the truth and can see exactly what each measure does. In a real
project, the coding is the expensive part; the code below is the easy part.

**Contents**

1. The codebook
2. A corpus, and a sample to annotate
3. Agreement between coders
4. Learning from disagreements
5. Validating a classifier
6. How many documents to annotate?
7. Proportions: when accuracy isn't enough
8. LLMs as annotators, tools and reporting

No GPU needed.

## 0. Setup

In [ ]:
!pip install -q krippendorff

import random

import krippendorff
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (classification_report, cohen_kappa_score, confusion_matrix,
                             ConfusionMatrixDisplay, f1_score)
from statsmodels.stats.proportion import proportion_confint

rng = np.random.default_rng(7)
pd.set_option("display.max_colwidth", 110)

## 1. The codebook

Our running example: the **dominant frame** of sentences from news coverage of Dutch nitrogen policy.
A codebook turns a concept into instructions that two people can apply the same way. For each
category it gives a definition, examples, and, most importantly, **decision rules** for the
borderline cases.

| Category | Definition | Example |
|---|---|---|
| `economy` | costs, incomes, jobs, prices, compensation | *"Dairy farmers fear for their income."* |
| `environment` | nature, emissions, biodiversity, health effects | *"Nitrogen deposition is damaging heathland."* |
| `politics` | parties, elections, coalitions, protests aimed at politicians | *"The coalition is divided over the deadline."* |
| `human interest` | individual people and their personal stories | *"For Gerrit, the farm is three generations of family history."* |

**Decision rules** (the part coders actually need):

1. Code the frame of the **main clause**. *"Farmers fear for their income as nature rules tighten"*
   → `economy`.
2. A protest is `politics` if it targets a political decision, `human interest` if the sentence is
   about one person's experience.
3. If no rule decides, choose the frame that the sentence would most likely be quoted for.

Codebooks are written in rounds: draft, code a small pilot sample together, discuss disagreements,
sharpen the rules, repeat until agreement is good enough. Keep every version; the changes are part of
your method.

## 2. A corpus, and a sample to annotate

We generate 3,000 sentences from two (made-up) outlets with a known true frame. About one in five is
deliberately **ambiguous**: it mixes two frames, so even careful coders will sometimes disagree. We
also add the predictions of an imperfect automatic classifier, which we'll validate later.

In [ ]:
FRAMES = ["economy", "environment", "politics", "human interest"]
PHRASES = {
    "economy": ["dairy farmers fear for their income", "the buyout scheme will cost billions",
                "land prices in the province keep rising", "feed suppliers expect to lose jobs",
                "the compensation per cow was raised"],
    "environment": ["nitrogen deposition is damaging heathland", "ammonia emissions fell for the third year",
                    "protected species are disappearing from the reserve", "researchers measured high nitrate levels",
                    "the Veluwe forests show clear signs of stress"],
    "politics": ["the coalition is divided over the deadline", "the farmers' party won the provincial vote",
                 "the minister survived a vote of no confidence", "MPs demanded a new nitrogen plan",
                 "tractors blocked the road to parliament"],
    "human interest": ["for one farmer the farm is three generations of history",
                       "a young couple gave up their dream of taking over the farm",
                       "a retired farmer remembers the first protests",
                       "a farmer's daughter tells how her father sleeps badly",
                       "a vet describes her nights on the road"],
}
OPENERS = ["According to the report,", "On Tuesday", "Critics say", "In the east of the country", "Once again"]

def make_sentence(frame, ambiguous):
    main = rng.choice(PHRASES[frame])
    if ambiguous:                                 # add a clause from another frame
        other = rng.choice([f for f in FRAMES if f != frame])
        return f"{rng.choice(OPENERS)} {main}, while {rng.choice(PHRASES[other])}.", other
    return f"{rng.choice(OPENERS)} {main}.", None

N = 3000
true = rng.choice(FRAMES, size=N, p=[0.30, 0.35, 0.25, 0.10])
outlet = rng.choice(["Agri Weekly", "City Daily"], size=N, p=[0.4, 0.6])
ambiguous = rng.random(N) < 0.2
made = [make_sentence(f, a) for f, a in zip(true, ambiguous)]
corpus = pd.DataFrame({"id": np.arange(N), "outlet": outlet, "text": [m[0] for m in made],
                       "true": true, "ambiguous": ambiguous, "second_frame": [m[1] for m in made]})

# An imperfect classifier: right about 80% of the time, worse on ambiguous sentences, and when it's
# wrong it tends to see economy, especially in the farming outlet (a kind of error that matters later)
def classify(row):
    p_right = 0.62 if row.ambiguous else 0.86
    if rng.random() < p_right:
        return row.true
    if row.true != "economy" and rng.random() < (0.8 if row.outlet == "Agri Weekly" else 0.3):
        return "economy"
    if isinstance(row.second_frame, str):            # confuse it with the sentence's other frame
        return row.second_frame
    return rng.choice([f for f in FRAMES if f != row.true])

corpus["predicted"] = corpus.apply(classify, axis=1)
corpus.head(8)

In real life, you know neither `true` nor whether a sentence is ambiguous. The point of annotation is
to *estimate* `true` for a sample, so that you can judge `predicted`.

### Drawing the sample

- A **simple random sample** gives unbiased estimates for the whole corpus. Use it for validation.
- With a **rare category**, a random sample contains few examples of it. You can oversample it
  (stratify, e.g., by the classifier's prediction), but then you must weight the results to say
  anything about the whole corpus.
- **Blind** the coders: don't show them the classifier's prediction, the outlet, or each other's
  codes.
- Shuffle the order, so coders don't drift systematically through the corpus.

Here we draw a pilot sample of 200 for testing the codebook, and a separate validation sample of 400,
and write the coding sheet a coder would get:

In [ ]:
shuffled = corpus.sample(frac=1, random_state=1)
pilot = shuffled.iloc[:200].copy()
validation = shuffled.iloc[200:600].copy()

coding_sheet = validation[["id", "text"]].assign(frame="", notes="")
coding_sheet.to_csv("coding_sheet_validation.csv", index=False)
coding_sheet.head(3)

### Simulating coders

Instead of real people, we simulate three coders on the pilot sample. Coders A and B are careful
but human: right about 97% of the time on clear sentences and 80% on ambiguous ones, where their
mistakes go to the *other* frame in the sentence. Coder C has misunderstood the codebook and often
codes environmental sentences as politics.

In [ ]:
def simulate_coder(df, p_clear=0.97, p_ambiguous=0.80, confuse=None, seed=0):
    r = np.random.default_rng(seed)
    codes = []
    for row in df.itertuples():
        if confuse and row.true == confuse[0] and r.random() < confuse[2]:
            codes.append(confuse[1]); continue                # systematic misunderstanding
        p = p_ambiguous if row.ambiguous else p_clear
        if r.random() < p:
            codes.append(row.true)
        else:
            other = row.second_frame if isinstance(row.second_frame, str) else None   # NaN if none
            codes.append(other or r.choice([f for f in FRAMES if f != row.true]))
    return np.array(codes)

pilot["A"] = simulate_coder(pilot, seed=1)
pilot["B"] = simulate_coder(pilot, seed=2)
pilot["C"] = simulate_coder(pilot, seed=3, confuse=("environment", "politics", 0.4))
pilot[["text", "A", "B", "C"]].head(6)

## 3. Agreement between coders

**Percent agreement** is the share of items on which two coders chose the same code. It's intuitive,
but it rewards luck: two coders who guess at random still agree sometimes, and the more skewed the
categories, the more often.

**Cohen's kappa** corrects for that:

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

where $p_o$ is the observed agreement and $p_e$ the agreement expected by chance, given how often
each coder uses each category. $\kappa = 1$ is perfect, $0$ is chance level.

**Krippendorff's alpha** is the most general: any number of coders, missing codes, and nominal,
ordinal or interval data. It's the standard in communication science. Common benchmarks
(Krippendorff, 2004): $\alpha \geq .800$ for reliable data, $.667 \leq \alpha < .800$ for tentative
conclusions only.

In [ ]:
def percent_agreement(a, b):
    return np.mean(np.asarray(a) == np.asarray(b))

def alpha(*coders):
    # Krippendorff's alpha (nominal) for any number of coders; codes as strings, missing as None
    cats = {c: i for i, c in enumerate(FRAMES)}
    data = [[cats.get(c, np.nan) if c is not None else np.nan for c in coder] for coder in coders]
    return krippendorff.alpha(reliability_data=np.array(data, dtype=float), level_of_measurement="nominal")

print(f"A vs B:  agreement {percent_agreement(pilot.A, pilot.B):.2f}   "
      f"kappa {cohen_kappa_score(pilot.A, pilot.B):.2f}   alpha {alpha(pilot.A, pilot.B):.2f}")

> **✏️ Exercise 1**
>
> Compute percent agreement, Cohen's kappa and Krippendorff's alpha for **A vs C** and **B vs C**, and
> alpha for all three coders together. Which coder is the problem? Would percent agreement alone
> have told you clearly?

In [ ]:
# ✅ Solution
for x, y in [("A", "B"), ("A", "C"), ("B", "C")]:
    print(f"{x} vs {y}:  agreement {percent_agreement(pilot[x], pilot[y]):.2f}   "
          f"kappa {cohen_kappa_score(pilot[x], pilot[y]):.2f}   alpha {alpha(pilot[x], pilot[y]):.2f}")
print(f"A, B, C together: alpha {alpha(pilot.A, pilot.B, pilot.C):.2f}")
# C agrees less with both others, so C is the problem. Percent agreement still looks respectable for
# the pairs with C, which is exactly why it isn't enough: kappa and alpha make the gap clearer by
# discounting the agreement you'd get anyway.

### Why percent agreement misleads

Imagine a rare category, like "mentions a specific farmer by name", in about 5% of sentences. Two
coders who *never* look at the text and just say "no" almost always agree:

In [ ]:
lazy_a = np.array(["no"] * 200)
real = np.where(rng.random(200) < 0.05, "yes", "no")       # a coder who actually reads
print(f"Percent agreement: {percent_agreement(lazy_a, real):.2f}")
print(f"Cohen's kappa:     {cohen_kappa_score(lazy_a, real):.2f}")

More than 90% agreement, zero information. Always report a chance-corrected measure, and for rare categories,
also check agreement on the positive cases specifically.

### Missing codes

In real projects, not every coder codes every item (coders overlap only partly to save time).
Krippendorff's alpha handles that directly: missing codes are simply left out of the comparison.

> **✏️ Exercise 2**
>
> Suppose each coder coded only a random 70% of the pilot sample. Set the other 30% of each coder's
> codes to `None` (independently per coder), compute alpha for A, B and C with the missing codes, and
> compare it with alpha on only the items that all three coded. How many items does each version use?

In [ ]:
# ✅ Solution
masked = {}
for c, seed in [("A", 11), ("B", 12), ("C", 13)]:
    keep = np.random.default_rng(seed).random(len(pilot)) < 0.7
    masked[c] = [code if k else None for code, k in zip(pilot[c], keep)]

print(f"alpha with missing codes:    {alpha(masked['A'], masked['B'], masked['C']):.2f}  "
      f"(items coded by at least 2: {sum(sum(m[i] is not None for m in masked.values()) >= 2 for i in range(len(pilot)))})")
complete = [i for i in range(len(pilot)) if all(masked[c][i] is not None for c in masked)]
print(f"alpha on complete items only: {alpha(*[[masked[c][i] for i in complete] for c in masked]):.2f}  "
      f"(items: {len(complete)})")
# Similar values, but the complete-case version throws away most of the data (0.7^3 = only about a third
# of the items were coded by all three). Alpha with missing codes uses every item with 2+ codes.

## 4. Learning from disagreements

A single agreement number says *how much* coders disagree, not *where*. A confusion matrix between two
coders shows which categories get mixed up, which tells you which decision rule to sharpen.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (x, y) in zip(axes, [("A", "B"), ("A", "C")]):
    ConfusionMatrixDisplay.from_predictions(pilot[x], pilot[y], labels=FRAMES, ax=ax, colorbar=False,
                                            cmap="Greens", xticks_rotation=30)
    ax.set_xlabel(f"coder {y}"); ax.set_ylabel(f"coder {x}")
    ax.set_title(f"{x} vs {y}")
plt.tight_layout(); plt.show()

A vs B shows scattered disagreement, mostly on ambiguous sentences: that's normal noise. A vs C shows a
**pattern**: many sentences that A codes `environment` are `politics` for C. That's a codebook
problem (or a training problem), not noise. Talk to C, look at the examples together, and add a rule.

Let's check where A and B disagree:

In [ ]:
disagree = pilot[pilot.A != pilot.B]
print(f"{len(disagree)} disagreements; {disagree.ambiguous.mean():.0%} of them on ambiguous sentences "
      f"(vs {pilot.ambiguous.mean():.0%} ambiguous overall)")
disagree[["text", "A", "B"]].head(6)

### From two coders to one gold standard

For the validation sample, two coders each code everything; then the disagreements are resolved by
**discussion** or by a third person (**adjudication**). The result is the gold standard. Report the
agreement *before* adjudication: that's the reliability of your coding procedure.

In [ ]:
validation["A"] = simulate_coder(validation, seed=21)
validation["B"] = simulate_coder(validation, seed=22)
print(f"Validation sample, A vs B: alpha {alpha(validation.A, validation.B):.2f}")

# Adjudication: where A and B agree, take their code; where they disagree, a third person decides
# (simulated: right 85% of the time, otherwise sides with A or B)
adj_rng = np.random.default_rng(23)
validation["gold"] = [a if a == b else (t if adj_rng.random() < 0.85 else adj_rng.choice([a, b]))
                      for a, b, t in zip(validation.A, validation.B, validation.true)]
print(f"The gold standard matches the (normally unknown) truth for {np.mean(validation.gold == validation.true):.1%} of items")

## 5. Validating a classifier

Now we compare the classifier's predictions with the gold standard: the measures from Day 2.

In [ ]:
print(classification_report(validation.gold, validation.predicted, labels=FRAMES, digits=2))

Two things matter beyond the table.

**Uncertainty.** 400 documents is a sample, so these numbers have margins of error. For accuracy (a
proportion), the **Wilson interval** works well; for F1, a **bootstrap** (re-sample the validation
set many times and look at the spread).

In [ ]:
correct = (validation.predicted == validation.gold)
low, high = proportion_confint(correct.sum(), len(correct), alpha=0.05, method="wilson")
print(f"Accuracy: {correct.mean():.3f}   95% CI [{low:.3f}, {high:.3f}]")

boot = []
for _ in range(1000):
    idx = rng.integers(0, len(validation), len(validation))
    boot.append(f1_score(validation.gold.values[idx], validation.predicted.values[idx],
                         labels=["human interest"], average="macro", zero_division=0))
print(f"F1 for 'human interest': {f1_score(validation.gold, validation.predicted, labels=['human interest'], average='macro'):.2f}"
      f"   95% bootstrap CI [{np.percentile(boot, 2.5):.2f}, {np.percentile(boot, 97.5):.2f}]")

The rare category gets a much wider interval: only about 40 of the 400 sentences are `human interest`.

**Subgroups.** A classifier that works well overall can work badly for part of the data, for example
one outlet, one period or one language. If your analysis compares groups, validate per group:

In [ ]:
per_outlet = (validation.assign(correct=correct)
              .groupby("outlet")
              .agg(n=("correct", "size"), accuracy=("correct", "mean"),
                   predicted_economy=("predicted", lambda s: (s == "economy").mean()),
                   gold_economy=("gold", lambda s: (s == "economy").mean())))
per_outlet.round(3)

In *Agri Weekly*, the classifier over-predicts `economy`. Any comparison of the economy frame
between the two outlets would be biased, even though overall accuracy looks fine.

## 6. How many documents to annotate?

It depends on what you want to know, and how precisely. The width of a confidence interval shrinks
with the square root of the sample size: to halve it, you need four times as many documents.

In [ ]:
ns = [50, 100, 200, 400, 800, 1600]
widths = [np.diff(proportion_confint(int(0.8 * n), n, method="wilson"))[0] / 2 for n in ns]
plt.figure(figsize=(7, 3))
plt.plot(ns, widths, "o-", color="#34B233")
for n, w in zip(ns, widths):
    plt.annotate(f"±{w:.3f}", (n, w), textcoords="offset points", xytext=(4, 6), fontsize=8)
plt.xscale("log"); plt.xticks(ns, ns)
plt.xlabel("validation documents"); plt.ylabel("margin of error")
plt.title("95% margin of error for an accuracy of 0.80")
plt.show()

> **✏️ Exercise 3**
>
> 1. Find the smallest validation set (in steps of 25) for which the 95% Wilson margin of error of an
>    accuracy of 0.80 is at most ±0.03.
> 2. For a category that makes up 10% of the corpus, how many documents of that category would a
>    random sample of that size contain? What does that mean for the precision of its F1?

In [ ]:
# ✅ Solution
for n in range(25, 3001, 25):
    lo, hi = proportion_confint(int(round(0.8 * n)), n, method="wilson")
    if (hi - lo) / 2 <= 0.03:
        print(f"n = {n}: margin ±{(hi - lo) / 2:.3f}")
        break
print(f"A 10% category would have about {int(0.1 * n)} examples in it")
# Around 700 documents for ±3 points on accuracy. But a 10% category would get only ~70 examples, so
# its F1 stays quite uncertain. If rare categories matter, oversample them (and weight), or annotate
# more.

## 7. Proportions: when accuracy isn't enough

Often the goal isn't to classify single documents but to estimate a **proportion**: what share of
coverage uses the economy frame, and how that differs between outlets or changes over time. Simply
counting the classifier's predictions ("classify and count") is biased whenever the errors aren't
balanced.

Compare, for the whole corpus:

In [ ]:
share = pd.DataFrame({"true (unknown in practice)": corpus.true.value_counts(normalize=True),
                      "classify and count": corpus.predicted.value_counts(normalize=True)}).loc[FRAMES]
share.round(3)

For one category, there's a simple correction. Treat "economy or not" as a binary task and estimate
from the validation set how often the classifier says `economy` when it's true (the true positive
rate, TPR) and when it isn't (the false positive rate, FPR). Then

$$\hat{p}_{\text{corrected}} = \frac{\hat{p}_{\text{predicted}} - \text{FPR}}{\text{TPR} - \text{FPR}}$$

because $\hat{p}_{\text{predicted}} = \text{TPR} \cdot p + \text{FPR} \cdot (1 - p)$.

> **✏️ Exercise 4**
>
> 1. Estimate TPR and FPR for `economy` from the validation sample (gold vs predicted), and apply
>    the correction to the share of `economy` predictions in the whole corpus. Compare with the naive
>    count and with the truth.
> 2. Do the same **per outlet**. Is one overall correction enough, or do you need one per outlet?

In [ ]:
# ✅ Solution
def tpr_fpr(gold, pred, cat="economy"):
    gold, pred = np.asarray(gold) == cat, np.asarray(pred) == cat
    return pred[gold].mean(), pred[~gold].mean()

def corrected(p_pred, tpr, fpr):
    return float(np.clip((p_pred - fpr) / (tpr - fpr), 0, 1))

tpr, fpr = tpr_fpr(validation.gold, validation.predicted)
p_pred = (corpus.predicted == "economy").mean()
print(f"TPR {tpr:.2f}, FPR {fpr:.2f}")
print(f"economy share: naive {p_pred:.3f}, corrected {corrected(p_pred, tpr, fpr):.3f}, "
      f"truth {(corpus.true == 'economy').mean():.3f}")

rows = []
for name in ["Agri Weekly", "City Daily"]:
    v, c = validation[validation.outlet == name], corpus[corpus.outlet == name]
    p = (c.predicted == "economy").mean()
    t_o, f_o = tpr_fpr(v.gold, v.predicted)
    rows.append({"outlet": name, "naive": p, "overall correction": corrected(p, tpr, fpr),
                 "per-outlet correction": corrected(p, t_o, f_o), "truth": (c.true == "economy").mean()})
display(pd.DataFrame(rows).round(3))

# How sure are we about the outlet DIFFERENCE after per-outlet correction? Bootstrap the validation set.
p_out = {name: (corpus[corpus.outlet == name].predicted == "economy").mean() for name in ["Agri Weekly", "City Daily"]}
diffs = []
for _ in range(1000):
    v = validation.sample(frac=1, replace=True, random_state=int(rng.integers(1e9)))
    est = {name: corrected(p_out[name], *tpr_fpr(v[v.outlet == name].gold, v[v.outlet == name].predicted))
           for name in p_out}
    diffs.append(est["Agri Weekly"] - est["City Daily"])
print(f"Agri Weekly minus City Daily, corrected: 95% CI [{np.percentile(diffs, 2.5):+.3f}, {np.percentile(diffs, 97.5):+.3f}]")
print(f"Naive difference: {p_out['Agri Weekly'] - p_out['City Daily']:+.3f};  "
      f"true difference: {(corpus[corpus.outlet == 'Agri Weekly'].true == 'economy').mean() - (corpus[corpus.outlet == 'City Daily'].true == 'economy').mean():+.3f}")
# Naive counting makes Agri Weekly look more economy-focused than City Daily; in truth they hardly
# differ. The false positive rate differs between outlets, so one overall correction can't fix the
# comparison. Per-outlet corrections move both estimates toward the truth, but with only 150-250
# validation documents per outlet they are noisy: the bootstrap interval for the difference is wide
# and includes zero. Honest conclusion: no evidence of a difference. Bonus 12 takes this further.

## 8. LLMs as annotators, tools and reporting

### LLMs as annotators

Large language models can annotate text with surprisingly good accuracy, sometimes better than
crowd workers (Gilardi et al., 2023), and Day 4 shows how. Everything in this notebook applies to them
unchanged: an LLM is one more coder, or one more classifier, to be validated against a human gold
standard, **per task, per dataset, per prompt and per model version**. Treat a prompt like a codebook:
pilot it, look at the disagreements, revise, and freeze it before the final run.

> **✏️ Exercise 5**
>
> Treat the classifier as a fourth "coder" on the pilot sample: add its predictions as a column and
> compute alpha for A, B and the classifier, and for A and B alone. Would the classifier lower the
> reliability of a coding team? What does this comparison tell you, and what doesn't it?

In [ ]:
# ✅ Solution
print(f"alpha A, B:              {alpha(pilot.A, pilot.B):.2f}")
print(f"alpha A, B + classifier: {alpha(pilot.A, pilot.B, pilot.predicted):.2f}")
print(f"classifier vs A: kappa {cohen_kappa_score(pilot.A, pilot.predicted):.2f};  "
      f"vs B: kappa {cohen_kappa_score(pilot.B, pilot.predicted):.2f}")
# The classifier agrees with the humans less than they agree with each other, so adding it lowers
# alpha. "Does the machine agree with humans about as well as humans agree with each other?" is a
# useful benchmark. What it doesn't tell you: whether its errors are systematic (Section 5's outlet
# problem), which matters more for your conclusions than the overall level.

### Annotation tools

For a few hundred items, a spreadsheet works. For more, or for span annotation (marking words in a
text), use a tool:

- **Label Studio**, **Doccano**, **INCEpTION**: open source, can run locally
- **Prodigy**: paid, by the makers of spaCy, built for fast annotation with active learning

### What to report

- the codebook (as an appendix or online), and how it was developed
- how the sample was drawn, and its size
- coders: who, how trained, blind to what
- reliability **before** adjudication (Krippendorff's alpha, with the number of items and coders), per
  variable or category
- how disagreements were resolved
- validation of each automated measure against the gold standard, with confidence intervals, and per
  subgroup if you compare groups

### Further reading

- Gilardi, F., Alizadeh, M., & Kubli, M. (2023). ChatGPT outperforms crowd workers for text-annotation
  tasks. *PNAS, 120*(30), e2305016120.
- Grimmer, J., & Stewart, B. M. (2013). Text as data: The promise and pitfalls of automatic content
  analysis methods for political texts. *Political Analysis, 21*(3), 267-297.
- Hayes, A. F., & Krippendorff, K. (2007). Answering the call for a standard reliability measure for
  coding data. *Communication Methods and Measures, 1*(1), 77-89.
- Krippendorff, K. (2004). *Content analysis: An introduction to its methodology* (2nd ed.). Sage.
- Song, H., Tolochko, P., Eberl, J.-M., et al. (2020). In validations we trust? The impact of imperfect human annotations
  as a gold standard on the quality of validation of automated content analysis. *Political
  Communication, 37*(4), 550-572.